### Zoteroize and Obsidianize a Perplexity Dialogue

In a Perplexity dialogue saved by Perplexity itself, replace the citation numbers with matching Obsidian literature note or Zotero item links

**It's half done.**  If you want to process plain perplexity outputs like this code does, but well, it might make sense to hack the code for relinking `Save my Chatbot` exports.

In [4]:
import re
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import sys
# from urllib.parse import urlparse, urlunparse
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as relinker

%load_ext autoreload
%autoreload 2

In [3]:
tmp_dir = rfw.refwrangle_test_dir / 'tmp'

perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
output_file_perplex = tmp_dir / "tmp_new_cites_perplexity_example.md"  # processed raw perplexity output

##### get the URLs of all parent items in the zotero db, and find out which have obsidian literature notes


## For raw perplexity dialog markdown

#### R1-inspired version

This partly works, but it's only looking up stuff by URL. It's not getting the body link titles, it doesn't modify the citation table links, and it doesn't look up zotero items by title, as a fallback when the right entry in the zot db is missing a URL or as a different one for a paper with the same title

In [ ]:
from pathlib import Path
import re
import pandas as pd

def relink_perplexity_export(perplexity_doc: Path, output_file: Path) -> None:
    """Replace numeric citations with Obsidian/Zotero links while preserving original structure."""
    
    zotero_items = relinker.ZoteroLinkConverter().zotero_items
    
    url_to_zot_info = {row.url: row 
                       for row in zotero_items.itertuples(index=False)}

    # Read and split document
    content = perplexity_doc.read_text(encoding='utf-8')
    try:
        body, citations = content.split("\nCitations:\n", 1)
    except ValueError as e:
        # TODO: handle this
        raise ValueError("Invalid document structure - missing citations section") from e
    
    relinked_source_lines = []
    def make_link_from_source(cite_num: str, doc_url: str) -> str:

        numbered_url_link = f"[{cite_num}]({doc_url})"
        if zotero_item := relinker.find_zotero_item_via_url(doc_url):
            body_link = relinker.create_obsidian_or_zotero_link(zotero_item)
            relinked_source_lines.append(f'{numbered_url_link} **{body_link}**')
        else:
            body_ink = f"=={numbered_url_link}==" # mark it as "not in zotero"
            relinked_source_lines.append(f'{numbered_url_link} {doc_url}')
            
        return body_link

    source_matches = re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', citations, flags=re.M)
    source_num_to_link = {m.group('num'): make_link_from_source(m.group('num'), m.group('url')) for m in source_matches }
    
    # source_matches = re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', citations, flags=re.M)
    # source_num_to_url = {m.group('num'): rfw.normalize_url(m.group('url')) for m in source_matches }

    # def create_reference(doc_url: str, cite_num: str) -> str:
    #     """Generate appropriate reference link based on available metadata."""
    #     if not doc_url:
    #         return f'[{cite_num}]'
            
    #     item = url_to_zot_info.get(doc_url)
    #     if not item:
    #         return f'[{cite_num}]'

    #     if item.hasLitNote:
    #         return f'[[{item.citekey}]]'
            
    #     link_text = f"{item.citekey}→{item.zotkey[:6]}"
    #     return rfw.zotero_item_link(item.zotkey, link_text)

    # Process body content
    # body_relinked = re.sub(r'\[(\d+)\]',
    #                         lambda m: f' {create_reference(source_num_to_link.get(m.group(1)), m.group(1))}',
    #                         body)

    body_relinked = re.sub(r'\[(\d+)\]',lambda m: f' {source_num_to_link.get(m.group(1))}', body)
    sources_relinked = "\n".join(relinked_source_lines)
    
    # # Process citations section
    # def update_citation(m: re.Match) -> str:
    #     num = m.group('num')
    #     url = rfw.normalize_url(m.group('url'))
    #     if source_num_to
    #     if url in url_to_zot_info:
    #         # a hack to redo this here
    #         ref = create_reference(url, num)
    #         return f'[{num}] =={ref}== {m.group('url')}'
        
    #     return f'[{num}] {m.group('url')}'
    # def update_citation(m: re.Match) -> str:
    #     num = m.group('num')
    #     url = rfw.normalize_url(m.group('url'))
    #     if url in url_to_zot_info:
    #         # a hack to redo this here
    #         ref = create_reference(url, num)
    #         return f'[{num}] =={ref}== {m.group('url')}'
        
    #     return f'[{num}] {m.group('url')}'

    # citations_relinked = re.sub(
    #     r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)',
    #     update_citation,
    #     citations,
    #     flags=re.M
    # )

    # Write output
    output_file.write_text(
        f"{body_relinked}\nCitations:\n{sources_relinked}", 
        encoding='utf-8'
    )

In [8]:
relink_perplexity_export(perplexity_dialog_file, output_file_perplex)
ic(perplexity_dialog_file, output_file_perplex)
print('Done.')

Cache is valid. Reading data from cache.


ic| perplexity_dialog_file: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/perplexity_example.md')
    output_file_perplex: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_new_cites_perplexity_example.md')


Done.


#### "My" version

In [ ]:

# # Functions for replacing references in perplexity's dialog copy with links to existing obsidian notes or zotero items 

# def zotero_item_link(zotero_item_key, link_text):
#     """Makes a link to a zotero item, given its key"""
#     return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

# def normalize_url(url):
#     """Convert a URL to a standard form, so that it can be string-compared to the same URL
#     written by a different program, but which is also normalized by this function."""
#     parsed = urlparse(url.lower())
#     return urlunparse(parsed._replace(path=parsed.path.rstrip('/')))

# def replace_perplexity_dialogue_links(perplexity_doc, zot_db_items, output_file):
#     """Replace numeric citations in a perplexity dialog document with links to matching 
#     obsidian literature notes or to zotero items.  A 'match' is determined when the URL 
#     in the perplexity doc matches a zotero item's URL.  Link first to the obsidian literature note
#     when one exists, then try to link to a zotero item.  If neither is available don't change the link.

#     Arguments 
#     perplexity_doc: a full pathlib path to a file of markdown coming from perplexity's copy function
#     zot_db_items: a dataframe with a row of info for every zotero DB item.  The columns are: 
#         citekey: the obsidian note citekey (the stem of its filename)
#         zotkey: zoter item key
#         hasLitNote: true if an obsidian literature note already exists
#         url: the URL associated with this zotero DB item
#     output_file: a full pathlib path to where the output document should go"""

#     # organize the zotero DB info
#     if not isinstance(zot_db_items, pd.DataFrame):
#         raise Exception('Expected a dataframe.  Reading url_to_citekey from file does not yet handle new dataframe column')
#         df = pd.read_csv(zot_db_items) # assume it has url and citekey columns
#         zot_db_items = {normalize_url(url): citekey for url, citekey in zip(df.url, df.citekey)}

#     zot_db_items['url'] = zot_db_items['url'].apply(normalize_url)
#     zot_url_to_item_info = defaultdict(lambda: None, {url:info.iloc[0] for url, info in zot_db_items.groupby('url')})

#     # modify the perplexity dialog doc
#     with open(perplexity_doc, 'r') as mdfile:
#         content = mdfile.read()

#     # Split the content into body and citations
#     parts = content.split("\nCitations:\n")
#     if len(parts) != 2:
#         raise Exception("Couldn't find Citations section")
    
#     body, citations = parts

#     # From citations at doc bottom, get a url for each citation number
#     citation_urls = re.findall(r'\[(\d+)\]\s+(https?://\S+)', citations)
#     doc_number_to_url = defaultdict(lambda: None, {num:normalize_url(url) for num, url in citation_urls})

#     def make_best_reference_link(doc_url, doc_cite_num):
#         # Replace body citations w/ wikilinks to an obsidian note or if no note, an md link to a zotero item
#         if doc_url and (itemInfo := zot_url_to_item_info[doc_url]) is not None:
#             if itemInfo.hasLitNote:
#                 return f'[[{itemInfo.citekey}]]' # wikilink to obsidian lit note

#             # md link to item in zotero DB
#             link_text = f'{itemInfo.citekey}\u2794{itemInfo.zotkey}'  #"bob \u2794 jim"
#             return f'{zotero_item_link(itemInfo.zotkey, link_text)}'
            
#         return f'[{doc_cite_num}]' # not in zotero DB so leave unchanged

#     def replace_body_reference(match):
#         # Replace citations in the body text
#         doc_cite_num = match.group(1)
#         doc_url = doc_number_to_url[doc_cite_num]

#         return ' ' + make_best_reference_link(doc_url, doc_cite_num)

#     body = re.sub(r'\[(\d+)\]', replace_body_reference, body)

#     def replace_citations_reference(match):
#         # Replace citations in the Citations section
#         doc_cite_num = match.group(1)
#         url = match.group(2)
#         doc_url = normalize_url(url)

#         if zot_url_to_item_info[doc_url] is None:
#             return f'[{doc_cite_num}] {doc_url}' # not in zotero DB
#         else:
#             return f'[{doc_cite_num}] =={make_best_reference_link(doc_url, doc_cite_num)}== {url}'

#     citations = re.sub(r'\[(\d+)\]\s+(https?://\S+)', replace_citations_reference, citations)

#     with open(output_file, 'w') as outfile:
#         outfile.write(body + "\nCitations:\n" + citations)


In [ ]:
D = dict(bob=1, fred=2, jill=3)

None in D